# SQL Injection Demo

In [ ]:
import mysql.connector
import os

In [ ]:
# Connect to the database

mydb = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

print(mydb)

--------------------------------------------------------------------------------------------------------------------------------------------------------------

#### Datenbankanbindung mit Python Teil 1 (12)
SQL Injection

In [ ]:
# 1 SQL injection vulnerability Demo: exploiting logical

sql_inj_cursor = mydb.cursor()

userInput = "'Jonas' OR 1 = 1"  # 1 = 1 is always true

# Problem: die Query wird hier bereits zu einem vollständigen String zusammengesetzt und verwendet somit keine Parameter mehr!
query = f"SELECT Nname, Ort FROM Klient WHERE VName = {userInput}"
print(query)

sql_inj_cursor.execute(query)

for x in sql_inj_cursor:
    print(x)

In [ ]:
# 2 SQL injection vulnerability Demo: if you have already information about the database schema

userInput = '"X" UNION SELECT Nname, Vname, Strasse, PLZ, Ort FROM Klient'
query = f"SELECT Nname, Vname, Strasse, PLZ, Ort FROM Gläubiger WHERE Vname = {userInput}"
print(query)
sql_inj_cursor.execute(query)
rows = sql_inj_cursor.fetchall()

for row in rows:
    print(row)

In [ ]:
# 3 SQL injection vulnerability Demo: obtaining admin rights

userInput = "'admin' --"  # the rest of the query is commented out (Attention! The comment dashes must follow a blank)
password = "'1234'"

query = f"SELECT * FROM User WHERE username = {userInput} AND passwort = {password}"
#f"SELECT s.höhe, s.grund FROM Schuld s JOIN Klient k ON s.KlientenNr = k.KlientenNr WHERE k.VName = {userInput}"
print(query)

sql_inj_cursor.execute(query)

for x in sql_inj_cursor:
    print(x)

--------------------------------------------------------------------------------------------------------------------------------------------------------------

#### Datenbankanbindung mit Python Teil 1 (12)
Prepared Statements

In [ ]:
# 1 SQL injection prevention with placeholder %s

cursor = mydb.cursor()

userInput = "'Jonas' OR 1 = 1"
query = "SELECT * FROM Klient WHERE VName = %s"
print(query)

cursor.execute(query, (userInput,))
rows = cursor.fetchall()

if rows:
    for x in rows:
        print(x)
else:
    print("Keine Daten gefunden")

In [ ]:
# 2 Why doesn't the prepared cursor impede the SQL injection?

prepared_cursor = mydb.cursor(prepared=True) # Der prepared Cursor trennt Parameter und Query auf Protokollebene.

userInput = "'Jonas' OR 1 = 1"
query = f"SELECT * FROM Klient WHERE VName = {userInput}"   # Weil die Query hier bereits zu einer vollständigen Query zusammengesetzt wird.
print(query)

prepared_cursor.execute(query)

for x in prepared_cursor:
    print(x)

In [ ]:
# 3 SQL injection prevention with prepared cursor and separation of query and parameter

userInput = "'Jonas' OR 1 = 1"
prepared_cursor.execute(f"SELECT * FROM Klient WHERE VName = ?", (userInput,))
rows = prepared_cursor.fetchall()

if rows:
    for x in rows:
        print(x)
else:
    print("Keine Daten gefunden")

### Best Practice: Prepared Cursor & %-Placeholder

In [ ]:
# 4 SQL injection prevention with prepared cursor and placeholder %s

prepared_cursor = mydb.cursor(prepared=True)

userInput = "'Jonas' OR 1 = 1"
query = "SELECT * FROM Klient WHERE VName = %s"
print(query)

prepared_cursor.execute(query, (userInput,))
rows = prepared_cursor.fetchall()

if rows:
    for x in rows:
        print(x)
else:
    print("Keine Daten gefunden")